In [ ]:
import os
from glob import glob
import geopandas
import pandas
import subprocess
from pathlib import Path
import sys
import numpy
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

# !{sys.executable} -m pip install "nismod-snail==0.5.3"

root = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import robyns_libraries.vector_raster_intersections



In [ ]:
# processed_data_path = 'L:\Jamaica\Inputs'
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_path = base_path / "dphil_paper_3/results_coastal_set6"
processed_data_path = base_path / "dphil_paper_3/processed_data"
networks_path = base_path / "dphil_common_cross_cutting/common_incoming_data/networks/networks"
damage_curves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/damage_curves"
robyn_libraries_path = base_path / "robyns_libraries"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"

jamaica_crs = 3448
jamaica_metric_grid_crs = "EPSG:3448"

mangroves_shapefile_candidates = [
    base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp",
    data_root / "landcover/mangroves_fn/mangroves.shp",
]
mangroves_shapefile = next((path for path in mangroves_shapefile_candidates if path.exists()), None)
if mangroves_shapefile is None:
    raise FileNotFoundError(
        "Could not find mangroves shapefile. Checked:\n" + "\n".join(str(path) for path in mangroves_shapefile_candidates)
    )

def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root / relative_asset_path
    asset_file_in_nested_networks_folder = data_root / "networks" / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: \n"
        f"- {asset_file_in_common_incoming_data}\n"
        f"- {asset_file_in_nested_networks_folder}"
    )

# base_path = 'Z:\\jamaica\\Inputs'
# output_path = 'Z:\\jamaica\\Results'


In [ ]:
network_csv = data_root / "networks/network_layers_hazard_intersections_details.csv" 
hazard_csv = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
damage_curves_csv = damage_curves_path / "asset_damage_curve_mapping.csv"
hazard_damage_parameters_csv = damage_curves_path / "hazard_damage_parameters.csv"
damage_results_folder = base_path / "dphil_paper_3/processed_data/direct_damages"


# network_csv = os.path.join(processed_data_path,
#                             "networks",
#                             "network_layers_hazard_intersections_details.csv")
# hazard_csv = os.path.join(processed_data_path,
#                             "coastal_flood_rasters.csv")
# damage_curves_csv = os.path.join(processed_data_path,
#                             "damage_curves",
# #                             "asset_damage_curve_mapping.csv")
# hazard_damage_parameters_csv = os.path.join(processed_data_path,
#                             "damage_curves",
#                             "hazard_damage_parameters.csv")
# damage_results_folder = "direct_damages"


In [ ]:
# Create a path to store intersection outputs
#output_path = os.path.join(output_path,"coastal_flood_intersections")
#if os.path.exists(output_path) == False:
    #os.mkdir(output_path)


In [ ]:
vector_details_csv = data_root /"networks/network_layers.csv"
raster_details_csv = data_root /"networks/hazard_layers.csv"


In [ ]:
# vector_details_csv = os.path.join(base_path,"infrastructure","network_layers.csv")
# raster_details_csv = os.path.join(base_path,"coastal_floods_FN","hazard_layers.csv")


In [ ]:

project_data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
results_directory = Path(output_path)
results_directory.mkdir(parents=True, exist_ok=True)

network_layers_input_file = project_data_root / "networks/network_layers_hazard_intersections_details.csv"
network_layers_table = pandas.read_csv(network_layers_input_file)
network_layers_table = network_layers_table[["path"]].drop_duplicates().reset_index(drop=True)
network_layers_table["path"] = network_layers_table["path"].str.replace(
    r"^networks/", "networks/networks/", regex=True
)
network_layers_output_file = results_directory / "network_layers_fixed_for_intersections.csv"
network_layers_table.to_csv(network_layers_output_file, index=False)

coastal_rasters_input_file = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
coastal_rasters_table = pandas.read_csv(coastal_rasters_input_file)
coastal_rasters_table["fname"] = coastal_rasters_table["path"]  # required by vector_raster_intersections.py
coastal_rasters_table["hazard"] = "coastal"  # align with hazard_damage_parameters.csv
hazard_layers_output_file = results_directory / "coastal_flood_rasters_fixed_for_intersections.csv"
coastal_rasters_table.to_csv(hazard_layers_output_file, index=False)

vector_details_csv = network_layers_output_file
raster_details_csv = hazard_layers_output_file
hazard_csv = hazard_layers_output_file

print("Network layers file:", vector_details_csv)
print("Hazard layers file:", raster_details_csv)
print("Summary hazard file:", hazard_csv)


In [ ]:
run_intersections = True  # Set to True is you want to run this process
if run_intersections is True:
    args = [
            "python",
            str(base_path / "robyns_libraries/vector_raster_intersections.py"),
            f"{vector_details_csv}",
            f"{raster_details_csv}",
            f"{output_path}"
            ]
    print ("* Start the processing of vector-raster intersections")
    print (args)
    subprocess.run(args)
print ("* Done with the processing of vector-raster intersections")


In [ ]:
# run_intersections = True  # Set to True is you want to run this process
# if run_intersections is True:
#     args = [
#             "python",
#             vector_raster_intersections.py",
#             f"{vector_details_csv}",
#             f"{raster_details_csv}",
#             f"{output_path}"
#             ]
#     print ("* Start the processing of vector-raster intersections")
#     print (args)
#     subprocess.run(args)
# print ("* Done with the processing of vector-raster intersections")


In [ ]:
# file_name = os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.geoparquet")
# df = geopandas.read_parquet(file_name)
# df



hazard_layers_name = Path(raster_details_csv).stem
file_name = Path(output_path) / f"airport_polygon_splits__{hazard_layers_name}__areas.geoparquet"
df = geopandas.read_parquet(file_name)
df


In [ ]:
# df.to_file(os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.gpkg"), layer="area", driver="GPKG")

gpkg_name = Path(output_path) / f"airport_polygon_splits__{hazard_layers_name}__areas.gpkg"
df.to_file(gpkg_name, layer="area", driver="GPKG")


In [ ]:

script_path = base_path / "scripts/analysis/damage_calculations_coastal_sensitivity.py"
hazard_layers_name = Path(raster_details_csv).stem
damage_results_folder = Path(output_path) / "direct_damages"
damage_results_folder.mkdir(parents=True, exist_ok=True)

sensitivity_csv = Path(output_path) / "sensitivity_parameters.csv"
pandas.DataFrame(
    [{"cost_uncertainty_parameter": 0.3333333333333333, "damage_uncertainty_parameter": 1.0}]
).to_csv(sensitivity_csv, index=False)
print("Using coastal sensitivity set6: cost_uncertainty_parameter=0.3333333333333333, damage_uncertainty_parameter=1.0")

asset_data_details = pandas.read_csv(network_csv)

for asset_info in asset_data_details.itertuples():
    asset_file_from_data_root = data_root / asset_info.path
    asset_file_with_networks_prefix = data_root / "networks" / asset_info.path

    if asset_file_from_data_root.exists():
        asset_gpkg_file = asset_file_from_data_root
    elif asset_file_with_networks_prefix.exists():
        asset_gpkg_file = asset_file_with_networks_prefix
    else:
        raise FileNotFoundError(
            "Asset file not found at either expected location:\n"
            f"{asset_file_from_data_root}\n"
            f"{asset_file_with_networks_prefix}"
        )
    intersection_file = Path(output_path) / f"{asset_info.asset_gpkg}_splits__{hazard_layers_name}__{asset_info.asset_layer}.geoparquet"
    output_file = damage_results_folder / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}" / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    args = [
        "python", str(script_path),
        "--network-csv", str(network_csv),
        "--hazard-csv", str(hazard_csv),
        "--sensitivity-csv", str(sensitivity_csv),
        "--sensitivity-id", "0",
        "--asset-gpkg-file", str(asset_gpkg_file),
        "--asset-gpkg-label", str(asset_info.asset_gpkg),
        "--asset-layer", str(asset_info.asset_layer),
        "--damage-curve-mapping-csv", str(damage_curves_csv),
        "--damage-threshold-uplift-csv", str(hazard_damage_parameters_csv),
        "--damage-curves-dir", str(damage_curves_path),
        "--intersection", str(intersection_file),
        "--output-path", str(output_file),
    ]
    print(args)
    run_result = subprocess.run(args, capture_output=True, text=True)
    if run_result.returncode != 0:
        print(run_result.stdout)
        print(run_result.stderr)
        run_result.check_returncode()

print("Finished direct damage calculations")


In [ ]:
# """Next we call the summary scripts
# """
# args = [
#         "python",
#         "damage_calculations.py",
#         f"{damage_results_folder}",
#         f"{network_csv}",
#         f"{hazard_csv}",
#         f"{damage_curves_csv}",
#         f"{hazard_damage_parameters_csv}",
#         "0","0","0"
#         ]
# print ("* Start the processing of summarising damage results")
# print (args)
# subprocess.check_output(args)


In [ ]:
damage_results_folder = os.path.join(output_path, "direct_damages")
mangrove_flood_damage_columns = ["coastal_flood_fn_mg_rp_25",
                       "coastal_flood_fn_mg_rp_100",
                       "coastal_flood_fn_mg_rp_500"]
nomangrove_flood_damage_columns = ["coastal_flood_fn_nomg_rp_25",
                       "coastal_flood_fn_nomg_rp_100",
                       "coastal_flood_fn_nomg_rp_500"]
difference_columns = ["coastal_flood_diff_rp_25",
                       "coastal_flood_diff_rp_100",
                       "coastal_flood_diff_rp_500"]
flood_damage_columns = mangrove_flood_damage_columns + nomangrove_flood_damage_columns + difference_columns 


In [ ]:
asset_data_details = pandas.read_csv(network_csv)
damage_totals = [] # List object to assemble many dataframes 
damage_estimates_directory = Path(output_path) / "damage_estimates"
damage_estimates_directory.mkdir(parents=True, exist_ok=True)
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_gpkg = asset_info.asset_gpkg
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        if os.path.exists(df_path):
            df = pandas.read_parquet(df_path) #read in the files using the .parquet from Raghav's code
            df[difference_columns] = df[nomangrove_flood_damage_columns] - df[mangrove_flood_damage_columns].values
            df = df.groupby([asset_id]).sum(flood_damage_columns).reset_index() # calculate asset level damages = .groupby(node_id).sum()
            asset_relative_path = asset_info.path
            resolved_asset_file = resolve_network_asset_file(asset_relative_path)
            df_geom = geopandas.read_file(resolved_asset_file, layer=asset_layer)
            df_geom = df_geom.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
            df = pandas.merge(df,df_geom[[asset_id,"geometry"]],how="left",on=[asset_id])
            df = geopandas.GeoDataFrame(df,geometry="geometry",crs=jamaica_crs)
            print(df)
            output_geopackage = damage_estimates_directory / f"{asset_gpkg}_{asset_layer}_asset_damages_groupedby.gpkg"
            df.to_file(output_geopackage, driver="GPKG")
            df['sector'] = asset_gpkg
            df['layer'] = asset_layer
            df = df.groupby(['sector','layer']).sum(flood_damage_columns).reset_index()
            damage_totals.append(df) # Add things to list

# Convert list of dataframes to 1 dataframe by concatenation
damage_totals = pandas.concat(damage_totals,axis=0,ignore_index=True)
damage_totals.to_csv(damage_estimates_directory / "asset_damages_groupedby.csv")


## Post-processing (clean and reproducible)
The cells below replace old duplicate/debug cells and give reproducible outputs.

In [ ]:
damage_estimates_directory = Path(output_path) / "damage_estimates"
if not damage_estimates_directory.exists():
    raise FileNotFoundError(f"Missing folder: {damage_estimates_directory}")

damage_estimate_files = sorted(damage_estimates_directory.glob("*_asset_damages_groupedby.gpkg"))
if not damage_estimate_files:
    raise FileNotFoundError(f"No grouped damage GPKGs found in {damage_estimates_directory}")

summary_rows = []
for grouped_damage_file in damage_estimate_files:
    grouped_damage = geopandas.read_file(grouped_damage_file)
    asset_name = grouped_damage_file.stem.replace("_asset_damages_groupedby", "")
    summary_rows.append({
        "asset_name": asset_name,
        "row_count": len(grouped_damage),
        "column_count": len(grouped_damage.columns),
    })

pandas.DataFrame(summary_rows).sort_values("asset_name").reset_index(drop=True)


In [ ]:
# Choose one output to inspect (this reproduces table outputs like the old debug cells).
asset_name_to_preview = "pipelines_NWC_edges"  # e.g. rail_nodes, roads_edges, buildings_assigned_economic_activity_areas
preview_file = damage_estimates_directory / f"{asset_name_to_preview}_asset_damages_groupedby.gpkg"

if not preview_file.exists():
    raise FileNotFoundError(f"Missing file: {preview_file}")

preview_table = geopandas.read_file(preview_file)
print(f"Rows: {len(preview_table)} | Columns: {len(preview_table.columns)}")
preview_table.head(20)


In [ ]:
# Reproducible mangrove-to-asset mapping for one asset layer.
asset_gpkg_for_mapping = "roads"
asset_layer_for_mapping = "edges"
asset_data_details = pandas.read_csv(network_csv)
asset_row = asset_data_details.loc[
    (asset_data_details.asset_gpkg == asset_gpkg_for_mapping)
    & (asset_data_details.asset_layer == asset_layer_for_mapping)
].squeeze()

if asset_row.empty:
    raise ValueError("No matching asset row found in network CSV")

asset_id_column = asset_row.asset_id_column
asset_file = resolve_network_asset_file(asset_row.path)
asset_geometry = geopandas.read_file(asset_file, layer=asset_layer_for_mapping)[[asset_id_column, "geometry"]]
asset_geometry = asset_geometry.to_crs(epsg=jamaica_crs)

mangroves_geometry = geopandas.read_file(mangroves_shapefile)[["ID", "geometry"]]
mangroves_geometry = mangroves_geometry.to_crs(epsg=jamaica_crs)

mangrove_asset_mapping = geopandas.sjoin(
    mangroves_geometry,
    asset_geometry,
    how="inner",
    predicate="intersects",
)[["ID", asset_id_column]].drop_duplicates().reset_index(drop=True)

print(f"Mapped pairs: {len(mangrove_asset_mapping)}")
if len(mangrove_asset_mapping) == 0:
    print("No intersections found for this asset selection. Try roads/edges or buildings_assigned_economic_activity/areas.")
mangrove_asset_mapping.head(30)


In [ ]:
# Sum damages by Sector, Subsector, and ReturnPeriod.
# Avoided damages are signed: Without_Mangroves - With_Mangroves (can be negative).
network_details = pandas.read_csv(network_csv)[["sector", "asset_description", "asset_gpkg", "asset_layer"]].drop_duplicates()

damage_rows = []
for asset_info in network_details.itertuples(index=False):
    damage_file = (
        Path(output_path)
        / "direct_damages"
        / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}"
        / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    )

    if not damage_file.exists():
        continue

    damage_table = pandas.read_parquet(damage_file)
    return_periods = sorted({col.split("_rp_")[-1] for col in damage_table.columns if "_rp_" in col})

    for return_period in return_periods:
        mg_col = f"coastal_flood_fn_mg_rp_{return_period}"
        nomg_col = f"coastal_flood_fn_nomg_rp_{return_period}"
        nomg_minus_mg_col = f"coastal_flood_fn_nomg_minus_mg_rp_{return_period}"

        damages_with_mangroves = float(damage_table[mg_col].sum()) if mg_col in damage_table.columns else 0.0

        if nomg_col in damage_table.columns:
            damages_without_mangroves = float(damage_table[nomg_col].sum())
        elif nomg_minus_mg_col in damage_table.columns:
            damages_without_mangroves = damages_with_mangroves + float(damage_table[nomg_minus_mg_col].sum())
        else:
            damages_without_mangroves = 0.0

        avoided_damages = damages_without_mangroves - damages_with_mangroves
        nomg_minus_mg_raster_damages = float(damage_table[nomg_minus_mg_col].sum()) if nomg_minus_mg_col in damage_table.columns else float('nan')

        damage_rows.append({
            "Sector": asset_info.sector,
            "Subsector": asset_info.asset_description,
            "Asset": asset_info.asset_gpkg,
            "Layer": asset_info.asset_layer,
            "ReturnPeriod": int(return_period),
            "Damages_With_Mangroves_JD": damages_with_mangroves,
            "Damages_Without_Mangroves_JD": damages_without_mangroves,
            "Avoided_Damages_JD": avoided_damages,
            "NomgMinusMg_Raster_Damages_JD": nomg_minus_mg_raster_damages,
        })

asset_level_summary = pandas.DataFrame(damage_rows)

sector_subsector_summary = (
    asset_level_summary
    .groupby(["Sector", "Subsector", "ReturnPeriod"], as_index=False)[
        ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD", "NomgMinusMg_Raster_Damages_JD"]
    ]
    .sum()
    .sort_values(["Sector", "Subsector", "ReturnPeriod"])
)

sector_summary = (
    sector_subsector_summary
    .groupby(["Sector", "ReturnPeriod"], as_index=False)[
        ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD", "NomgMinusMg_Raster_Damages_JD"]
    ]
    .sum()
    .sort_values(["Sector", "ReturnPeriod"])
)

sector_subsector_summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
sector_subsector_summary.to_csv(sector_subsector_summary_file, index=False)

print(f"Saved: {sector_subsector_summary_file}")
print("\nSector + Subsector summary:")
display(sector_subsector_summary)
print("\nSector-only summary:")
display(sector_summary)


In [ ]:
# Shared map inputs (loaded once)
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
airport_damage_file = Path(output_path) / "damage_estimates" / "airport_polygon_areas_asset_damages_groupedby.gpkg"

if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f"Missing Jamaica boundary file: {jamaica_boundary_path}")
jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)

if airport_damage_file.exists():
    airport_damage_base = geopandas.read_file(airport_damage_file).to_crs(jamaica_metric_grid_crs)
else:
    airport_damage_base = None


In [ ]:

# Draw roads using split coastal geometry, but color by edge-level avoided damages (J$).
roads_damage_file = Path(output_path) / "damage_estimates" / "roads_edges_asset_damages_groupedby.gpkg"
roads_splits_file = Path(output_path) / "roads_splits__coastal_flood_rasters_fixed_for_intersections__edges.geoparquet"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not roads_damage_file.exists():
    raise FileNotFoundError(f"Missing roads damage file: {roads_damage_file}")
if not roads_splits_file.exists():
    raise FileNotFoundError(f"Missing roads splits file: {roads_splits_file}")
roads_damage = geopandas.read_file(roads_damage_file)[["edge_id", avoided_damage_column]].copy()
roads_splits = geopandas.read_parquet(roads_splits_file)[["edge_id", "geometry"]].copy()
roads_splits = roads_splits.to_crs(jamaica_metric_grid_crs)

roads_segments = roads_splits.merge(roads_damage, on="edge_id", how="left")
roads_segments[avoided_damage_column] = roads_segments[avoided_damage_column].fillna(0.0)

zero_value_roads = roads_segments[numpy.abs(roads_segments[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_roads = roads_segments[roads_segments[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_roads = roads_segments[roads_segments[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_roads)} split road segments")
print(f"Negative avoided damages (red): {len(negative_value_roads)} split road segments")
print(f"Zero (white): {len(zero_value_roads)} split road segments")
if not roads_segments.empty:
    print(f"Min value: {roads_segments[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {roads_segments[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(11, 10))
axis.set_facecolor("#ffffff")
jamaica_boundary.boundary.plot(ax=axis, color="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_roads.empty:
    zero_value_roads.plot(ax=axis, color="#ffffff", linewidth=0.45, alpha=0.45, zorder=2)
if not positive_value_roads.empty:
    positive_value_roads.plot(ax=axis, color="#0b8f3f", linewidth=1.6, alpha=0.95, zorder=3)
if not negative_value_roads.empty:
    negative_value_roads.plot(ax=axis, color="#c81e1e", linewidth=1.6, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], color="#0b8f3f", lw=2.2, label="Avoided damages > 0 (green)"),
    Line2D([0], [0], color="#c81e1e", lw=2.2, label="Damages increase < 0 (red)"),
    Line2D([0], [0], color="#ffffff", lw=2.2, label="Zero change (white)"),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

axis.set_title(f"Road coastal-segment damage change map with Jamaica boundary (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"roads_coastal_segment_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

top_positive = roads_damage[["edge_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False).head(10)
top_negative = roads_damage[["edge_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=True).head(10)
print("\nTop 10 positive avoided damages (road edges):")
display(top_positive)
print("\nTop 10 negative avoided damages (road edges):")
display(top_negative)


In [ ]:

return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if airport_damage_base is None:
    raise FileNotFoundError(f"Missing airport damage file: {airport_damage_file}")
airport_damage = airport_damage_base.copy()
if avoided_damage_column not in airport_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

airport_damage = airport_damage.to_crs(jamaica_metric_grid_crs)
airport_damage = airport_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_airports = airport_damage[numpy.abs(airport_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_airports = airport_damage[airport_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_airports = airport_damage[airport_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_airports)} airports")
print(f"Negative avoided damages (red): {len(negative_value_airports)} airports")
print(f"Zero (white): {len(zero_value_airports)} airports")
if not airport_damage.empty:
    print(f"Min value: {airport_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {airport_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_airports.empty:
    zero_value_airports.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, zorder=2)
if not positive_value_airports.empty:
    positive_value_airports.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, zorder=3)
if not negative_value_airports.empty:
    negative_value_airports.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=12, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=12, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=12, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

axis.set_title(f"Airport damage change map with Jamaica boundary (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"airports_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(airport_damage[["node_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))


In [ ]:

return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if airport_damage_base is None:
    raise FileNotFoundError(f"Missing airport damage file: {airport_damage_file}")
airport_damage = airport_damage_base.copy()
if avoided_damage_column not in airport_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

airport_damage = airport_damage.to_crs(jamaica_metric_grid_crs)
airport_damage = airport_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_airports = airport_damage[numpy.abs(airport_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_airports = airport_damage[airport_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_airports = airport_damage[airport_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_airports)} airports")
print(f"Negative avoided damages (red): {len(negative_value_airports)} airports")
print(f"Zero (white): {len(zero_value_airports)} airports")
if not airport_damage.empty:
    print(f"Min value: {airport_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {airport_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_airports.empty:
    zero_value_airports.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, zorder=2)
if not positive_value_airports.empty:
    positive_value_airports.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, zorder=3)
if not negative_value_airports.empty:
    negative_value_airports.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=12, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=12, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=12, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

axis.set_title(f"Airport damage change map with Jamaica boundary (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"airports_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(airport_damage[["node_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))


In [ ]:

port_damage_file = Path(output_path) / "damage_estimates" / "port_polygon_areas_asset_damages_groupedby.gpkg"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not port_damage_file.exists():
    raise FileNotFoundError(f"Missing port damage file: {port_damage_file}")
port_damage = geopandas.read_file(port_damage_file)
if avoided_damage_column not in port_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

port_damage = port_damage.to_crs(jamaica_metric_grid_crs)
port_damage = port_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_ports = port_damage[numpy.abs(port_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_ports = port_damage[port_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_ports = port_damage[port_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_ports)} port polygons")
print(f"Negative avoided damages (red): {len(negative_value_ports)} port polygons")
print(f"Zero (white): {len(zero_value_ports)} port polygons")
if not port_damage.empty:
    print(f"Min value: {port_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {port_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_ports.empty:
    zero_value_ports.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, zorder=2)
if not positive_value_ports.empty:
    positive_value_ports.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, zorder=3)
if not negative_value_ports.empty:
    negative_value_ports.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=12, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=12, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=12, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

axis.set_title(f"Port damage change map with Jamaica boundary (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"ports_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(port_damage[["node_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))


In [ ]:

energy_nodes_damage_file = Path(output_path) / "damage_estimates" / "electricity_network_v3.1_nodes_asset_damages_groupedby.gpkg"
energy_edges_damage_file = Path(output_path) / "damage_estimates" / "electricity_network_v3.1_edges_asset_damages_groupedby.gpkg"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not energy_nodes_damage_file.exists():
    raise FileNotFoundError(f"Missing energy nodes damage file: {energy_nodes_damage_file}")
energy_nodes = geopandas.read_file(energy_nodes_damage_file)
if avoided_damage_column not in energy_nodes.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

energy_nodes = energy_nodes.to_crs(jamaica_metric_grid_crs)
energy_nodes = energy_nodes.dropna(subset=[avoided_damage_column]).copy()

zero_value_nodes = energy_nodes[numpy.abs(energy_nodes[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_nodes = energy_nodes[energy_nodes[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_nodes = energy_nodes[energy_nodes[avoided_damage_column] < -zero_tolerance_jd].copy()

energy_edges_rows = 0
if energy_edges_damage_file.exists():
    try:
        energy_edges = geopandas.read_file(energy_edges_damage_file)
        energy_edges_rows = len(energy_edges)
    except Exception:
        energy_edges_rows = 0

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_nodes)} energy nodes")
print(f"Negative avoided damages (red): {len(negative_value_nodes)} energy nodes")
print(f"Zero (white): {len(zero_value_nodes)} energy nodes")
print(f"Energy edges rows available: {energy_edges_rows}")
if not energy_nodes.empty:
    print(f"Min value: {energy_nodes[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {energy_nodes[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_nodes.empty:
    zero_value_nodes.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.6, alpha=0.75, markersize=55, zorder=2)
if not positive_value_nodes.empty:
    positive_value_nodes.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=1.2, alpha=0.95, markersize=70, zorder=3)
if not negative_value_nodes.empty:
    negative_value_nodes.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=1.2, alpha=0.95, markersize=70, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=10, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=10, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=10, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

axis.set_title(f"Energy (nodes) damage change map with Jamaica boundary (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"energy_nodes_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(energy_nodes[["id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False))


In [ ]:

buildings_damage_file = Path(output_path) / "damage_estimates" / "buildings_assigned_economic_activity_areas_asset_damages_groupedby.gpkg"
return_period_years = 100  # Change to 25, 100, or 500
zero_tolerance_jd = 0.0
avoided_damage_column = f"coastal_flood_diff_rp_{return_period_years}"

if not buildings_damage_file.exists():
    raise FileNotFoundError(f"Missing buildings damage file: {buildings_damage_file}")
buildings_damage = geopandas.read_file(buildings_damage_file)
if avoided_damage_column not in buildings_damage.columns:
    raise KeyError(f"Column not found: {avoided_damage_column}")

buildings_damage = buildings_damage.to_crs(jamaica_metric_grid_crs)
buildings_damage = buildings_damage.dropna(subset=[avoided_damage_column]).copy()

zero_value_buildings = buildings_damage[numpy.abs(buildings_damage[avoided_damage_column]) <= zero_tolerance_jd].copy()
positive_value_buildings = buildings_damage[buildings_damage[avoided_damage_column] > zero_tolerance_jd].copy()
negative_value_buildings = buildings_damage[buildings_damage[avoided_damage_column] < -zero_tolerance_jd].copy()

print(f"Return period: {return_period_years} years")
print(f"Positive avoided damages (green): {len(positive_value_buildings)} buildings")
print(f"Negative avoided damages (red): {len(negative_value_buildings)} buildings")
print(f"Zero (white): {len(zero_value_buildings)} buildings")
if not buildings_damage.empty:
    print(f"Min value: {buildings_damage[avoided_damage_column].min():,.2f} J$")
    print(f"Max value: {buildings_damage[avoided_damage_column].max():,.2f} J$")

figure, axis = plt.subplots(figsize=(10, 9))
axis.set_facecolor("#ffffff")
jamaica_boundary.plot(ax=axis, color="none", edgecolor="#bdbdbd", linewidth=0.45, zorder=1)

if not zero_value_buildings.empty:
    zero_value_buildings.plot(ax=axis, color="#ffffff", edgecolor="#b0b0b0", linewidth=0.2, alpha=0.65, zorder=2)
if not positive_value_buildings.empty:
    positive_value_buildings.plot(ax=axis, color="#0b8f3f", edgecolor="#0b8f3f", linewidth=0.2, alpha=0.88, zorder=3)
if not negative_value_buildings.empty:
    negative_value_buildings.plot(ax=axis, color="#c81e1e", edgecolor="#c81e1e", linewidth=0.2, alpha=0.88, zorder=4)

legend_handles = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#0b8f3f', markeredgecolor='#0b8f3f', markersize=10, label='Avoided damages > 0 (green)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#c81e1e', markeredgecolor='#c81e1e', markersize=10, label='Damages increase < 0 (red)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#ffffff', markeredgecolor='#b0b0b0', markersize=10, label='Zero change (white)'),
]
axis.legend(handles=legend_handles, loc="lower left", frameon=True, facecolor="white")

axis.set_title(f"Buildings damage change map with Jamaica boundary (RP {return_period_years})", fontsize=13)
axis.set_axis_off()
plt.tight_layout()

map_image_file = Path(output_path) / "damage_estimates" / f"buildings_damage_change_sign_map_with_boundary_rp_{return_period_years}.png"
figure.savefig(map_image_file, dpi=300, bbox_inches="tight")
print(f"Saved map image: {map_image_file}")
plt.show()

display(buildings_damage[["osm_id", avoided_damage_column]].sort_values(avoided_damage_column, ascending=False).head(30))


In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file)
sector_totals = (
    summary.groupby(["Sector", "ReturnPeriod"], as_index=False)["Avoided_Damages_JD"]
    .sum()
)

sector_order = ["buildings", "energy", "transport", "water"]
return_periods = [25, 100, 500]

def currency_formatter(x, _):
    return f"{x:,.0f}"

for rp in return_periods:
    rp_data = sector_totals[sector_totals["ReturnPeriod"] == rp].copy()
    rp_data["Sector"] = pandas.Categorical(rp_data["Sector"], categories=sector_order, ordered=True)
    rp_data = rp_data.sort_values("Sector")

    bar_colors = ["#0b8f3f" if value >= 0 else "#c81e1e" for value in rp_data["Avoided_Damages_JD"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(rp_data["Sector"], rp_data["Avoided_Damages_JD"], color=bar_colors, edgecolor="#2f2f2f", linewidth=0.6)
    ax.axhline(0, color="#7f7f7f", linewidth=0.8)
    ax.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
    ax.set_xlabel("Sector")
    ax.set_ylabel("Avoided damages (J$)")
    ax.set_title(f"Total sector avoided damages (RP {rp})")

    for bar, value in zip(bars, rp_data["Avoided_Damages_JD"]):
        y_pos = value if value >= 0 else value
        va = "bottom" if value >= 0 else "top"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            f"{value:,.0f}",
            ha="center",
            va=va,
            fontsize=9,
            rotation=0,
        )

    plt.tight_layout()
    output_chart = Path(output_path) / "damage_estimates" / f"sector_avoided_damages_bar_rp_{rp}.png"
    fig.savefig(output_chart, dpi=300, bbox_inches="tight")
    print(f"Saved: {output_chart}")
    plt.show()


In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file)
subsector_totals = (
    summary.groupby(["Sector", "Subsector", "ReturnPeriod"], as_index=False)["Avoided_Damages_JD"]
    .sum()
)

target_sectors = ["transport", "water"]
return_periods = [25, 100, 500]

def currency_formatter(x, _):
    return f"{x:,.0f}"

for sector_name in target_sectors:
    sector_data = subsector_totals[subsector_totals["Sector"] == sector_name].copy()
    subsector_order = sorted(sector_data["Subsector"].unique().tolist())

    for rp in return_periods:
        rp_data = sector_data[sector_data["ReturnPeriod"] == rp].copy()
        rp_data["Subsector"] = pandas.Categorical(rp_data["Subsector"], categories=subsector_order, ordered=True)
        rp_data = rp_data.sort_values("Subsector")

        bar_colors = ["#0b8f3f" if value >= 0 else "#c81e1e" for value in rp_data["Avoided_Damages_JD"]]

        fig, ax = plt.subplots(figsize=(11, 5.5))
        bars = ax.bar(rp_data["Subsector"], rp_data["Avoided_Damages_JD"], color=bar_colors, edgecolor="#2f2f2f", linewidth=0.6)
        ax.axhline(0, color="#7f7f7f", linewidth=0.8)
        ax.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax.set_xlabel("Subsector")
        ax.set_ylabel("Avoided damages (J$)")
        ax.set_title(f"{sector_name.capitalize()} subsector avoided damages (RP {rp})")
        plt.xticks(rotation=25, ha="right")

        for bar, value in zip(bars, rp_data["Avoided_Damages_JD"]):
            va = "bottom" if value >= 0 else "top"
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value,
                f"{value:,.0f}",
                ha="center",
                va=va,
                fontsize=8,
            )

        plt.tight_layout()
        output_chart = Path(output_path) / "damage_estimates" / f"{sector_name}_subsector_avoided_damages_bar_rp_{rp}.png"
        fig.savefig(output_chart, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_chart}")
        plt.show()


In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file).copy()

# Row-level share for each sector/subsector/return period record
summary["Avoided_Share_of_Baseline"] = numpy.where(
    summary["Damages_Without_Mangroves_JD"] > 0,
    summary["Avoided_Damages_JD"] / summary["Damages_Without_Mangroves_JD"],
    numpy.nan,
)
summary["Avoided_Share_of_Baseline_pct"] = 100 * summary["Avoided_Share_of_Baseline"]

summary_display_cols = [
    "Sector",
    "Subsector",
    "ReturnPeriod",
    "Damages_With_Mangroves_JD",
    "Damages_Without_Mangroves_JD",
    "Avoided_Damages_JD",
    "Avoided_Share_of_Baseline_pct",
]

print("Subsector-level avoided damages as % of baseline:")
display(
    summary[summary_display_cols]
    .sort_values(["Sector", "Subsector", "ReturnPeriod"])
    .round({"Avoided_Share_of_Baseline_pct": 2})
)

# Sector-level share uses ratio-of-sums (preferred aggregation)
sector_share = (
    summary.groupby(["Sector", "ReturnPeriod"], as_index=False)[
        ["Damages_With_Mangroves_JD", "Damages_Without_Mangroves_JD", "Avoided_Damages_JD"]
    ]
    .sum()
)
sector_share["Avoided_Share_of_Baseline"] = numpy.where(
    sector_share["Damages_Without_Mangroves_JD"] > 0,
    sector_share["Avoided_Damages_JD"] / sector_share["Damages_Without_Mangroves_JD"],
    numpy.nan,
)
sector_share["Avoided_Share_of_Baseline_pct"] = 100 * sector_share["Avoided_Share_of_Baseline"]

print("Sector-level avoided damages as % of baseline (ratio-of-sums):")
display(
    sector_share[
        [
            "Sector",
            "ReturnPeriod",
            "Damages_Without_Mangroves_JD",
            "Avoided_Damages_JD",
            "Avoided_Share_of_Baseline_pct",
        ]
    ]
    .sort_values(["Sector", "ReturnPeriod"])
    .round({"Avoided_Share_of_Baseline_pct": 2})
)

print("Quick comparison table (sector x return period, %):")
display(
    sector_share.pivot(index="Sector", columns="ReturnPeriod", values="Avoided_Share_of_Baseline_pct").round(2)
)

out_dir = Path(output_path) / "damage_estimates"
subsector_out = out_dir / "sector_subsector_return_period_damages_with_avoided_share.csv"
sector_out = out_dir / "sector_return_period_avoided_share.csv"

summary.to_csv(subsector_out, index=False)
sector_share.to_csv(sector_out, index=False)

print(f"Saved: {subsector_out}")
print(f"Saved: {sector_out}")


In [ ]:

summary_file = Path(output_path) / "damage_estimates" / "sector_subsector_return_period_damages.csv"
if not summary_file.exists():
    raise FileNotFoundError(f"Missing summary file: {summary_file}")

summary = pandas.read_csv(summary_file)
sector_share = (
    summary.groupby(["Sector", "ReturnPeriod"], as_index=False)[
        ["Damages_Without_Mangroves_JD", "Avoided_Damages_JD"]
    ]
    .sum()
)

sector_share["Avoided_Share_of_Baseline"] = numpy.where(
    sector_share["Damages_Without_Mangroves_JD"] > 0,
    sector_share["Avoided_Damages_JD"] / sector_share["Damages_Without_Mangroves_JD"],
    numpy.nan,
)
sector_share["Avoided_Share_of_Baseline_pct"] = 100 * sector_share["Avoided_Share_of_Baseline"]

sector_order = ["buildings", "energy", "transport", "water"]
return_periods = [25, 100, 500]

def pct_formatter(x, _):
    return f"{x:.1f}%"

for rp in return_periods:
    rp_data = sector_share[sector_share["ReturnPeriod"] == rp].copy()
    rp_data["Sector"] = pandas.Categorical(rp_data["Sector"], categories=sector_order, ordered=True)
    rp_data = rp_data.sort_values("Sector")

    bar_values = rp_data["Avoided_Share_of_Baseline_pct"].fillna(0)
    bar_colors = ["#0b8f3f" if value >= 0 else "#c81e1e" for value in bar_values]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(rp_data["Sector"], bar_values, color=bar_colors, edgecolor="#2f2f2f", linewidth=0.6)
    ax.axhline(0, color="#7f7f7f", linewidth=0.8)
    ax.yaxis.set_major_formatter(FuncFormatter(pct_formatter))
    ax.set_xlabel("Sector")
    ax.set_ylabel("Avoided damages (% of baseline)")
    ax.set_title(f"Sector avoided damages as % of baseline (RP {rp})")

    for bar, value in zip(bars, rp_data["Avoided_Share_of_Baseline_pct"]):
        if pandas.isna(value):
            label = "NA"
            y_pos = 0
            va = "bottom"
        else:
            label = f"{value:.2f}%"
            y_pos = value
            va = "bottom" if value >= 0 else "top"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            label,
            ha="center",
            va=va,
            fontsize=9,
        )

    plt.tight_layout()
    output_chart = Path(output_path) / "damage_estimates" / f"sector_avoided_share_bar_rp_{rp}.png"
    fig.savefig(output_chart, dpi=300, bbox_inches="tight")
    print(f"Saved: {output_chart}")
    plt.show()
